# ポッドキャスト ep001 音声生成iPhone だけで音声を作るためのノートブックです。**使い方: 上から順に、左の ▶ ボタンを押していくだけです。**各セルは前のセルが終わってから押してください（実行中は ▶ が回ります）。所要時間の目安: 全部で 20〜30 分。

## 1. 台本とツールを取得する

In [ ]:
!git clone --depth 1 -b claude/podcast-planning-daily-insights-2mbv5m https://github.com/noriishikawa323-coder/mori.git /content/mori%cd /content/mori!pip install -q -r podcast/requirements.txtprint("\n✅ 1 完了。次のセルへ。")

## 2. VOICEVOX エンジンを用意するいちばん時間がかかります（10 分ほど）。ダウンロード先は実行時に自動で探します。

In [ ]:
import json, os, re, subprocess, sys, time, urllib.request, glob, socketos.makedirs("/content/vv", exist_ok=True)def sh(cmd):    print("$", cmd)    return subprocess.run(cmd, shell=True).returncodesh("apt-get -qq install -y p7zip-full > /dev/null")# 最新リリースから Linux / CPU 版の資産を探すrel = json.load(urllib.request.urlopen(    "https://api.github.com/repos/VOICEVOX/voicevox_engine/releases/latest"))print("リリース:", rel["tag_name"])assets = [a for a in rel["assets"]          if "linux" in a["name"].lower() and "cpu" in a["name"].lower()]if not assets:    raise SystemExit("Linux/CPU 版が見つかりません。assets: "                     + ", ".join(a["name"] for a in rel["assets"]))# 分割書庫は全パートを落とすassets.sort(key=lambda a: a["name"])print("取得するファイル:")for a in assets:    print("  ", a["name"], f'{a["size"]/1e6:.0f}MB')for a in assets:    dst = "/content/vv/" + a["name"]    if not os.path.exists(dst):        sh(f'curl -sSfL -o "{dst}" "{a["browser_download_url"]}"')# 展開（.7z.001 があればそれを、無ければ .zip / .7z を）first = (sorted(glob.glob("/content/vv/*.7z.001"))         or sorted(glob.glob("/content/vv/*.7z"))         or sorted(glob.glob("/content/vv/*.zip")))if not first:    raise SystemExit("展開できる書庫が見つかりません: " + str(os.listdir("/content/vv")))sh(f'cd /content/vv && 7z x -y "{os.path.basename(first[0])}" > /dev/null')runs = [p for p in glob.glob("/content/vv/**/run", recursive=True) if os.path.isfile(p)]if not runs:    raise SystemExit("run が見つかりません: " + str(glob.glob("/content/vv/*")))run = runs[0]os.chmod(run, 0o755)print("エンジン:", run)subprocess.Popen([run, "--host", "127.0.0.1", "--port", "50021"],                 cwd=os.path.dirname(run),                 stdout=open("/content/vv/engine.log", "w"),                 stderr=subprocess.STDOUT)print("起動を待っています...", end="")for _ in range(180):    try:        with socket.create_connection(("127.0.0.1", 50021), timeout=1):            break    except OSError:        print(".", end=""); time.sleep(2)else:    print("\n起動しませんでした。/content/vv/engine.log の中身:")    print(open("/content/vv/engine.log").read()[-3000:])    raise SystemExit(1)ver = urllib.request.urlopen("http://127.0.0.1:50021/version").read().decode()print(f"\n✅ 2 完了。VOICEVOX {ver} が動いています。次のセルへ。")

## 3. セリフを合成する162 行を 1 行ずつ合成します（5〜15 分）。声を変えたいときは、下の `CASTING` を `"as_specified"` にするとずんだもん / 四国めたんになります。

In [ ]:
CASTING = "izakaya"   # "izakaya" = 玄野武宏 / 青山龍星 、 "as_specified" = ずんだもん / 四国めたん!python3 podcast/tools/synthesize.py --casting {CASTING} --forceprint("\n✅ 3 完了。次のセルへ。")

## 4. 1 本につないで音量を整える

In [ ]:
!python3 podcast/tools/build_episode.pyprint("\n✅ 4 完了。次のセルへ。")

## 5. iPhone に取り出すGoogle ドライブに保存します。実行すると認証を求められるので、画面の指示にしたがって自分の Google アカウントを許可してください。終わったら iPhone の **ドライブアプリ**（または ファイルアプリ）の`ポッドキャスト` フォルダに入っています。

In [ ]:
import shutil, osfrom google.colab import drivedrive.mount("/content/drive")dst = "/content/drive/MyDrive/ポッドキャスト"os.makedirs(dst, exist_ok=True)src = "podcast/audio/podcast_ep001.wav"shutil.copy(src, dst)shutil.copy("podcast/audio/podcast_ep001_cues.txt", dst)# 容量が軽い mp3 版も置いておく（スマホで聴くならこちらが楽）!python3 -c "import imageio_ffmpeg,subprocess;subprocess.run([imageio_ffmpeg.get_ffmpeg_exe(),'-v','error','-y','-i','podcast/audio/podcast_ep001.wav','-b:a','128k','/content/podcast_ep001.mp3'])"shutil.copy("/content/podcast_ep001.mp3", dst)print("\n✅ 完了しました。Google ドライブの「ポッドキャスト」フォルダを見てください。")for f in sorted(os.listdir(dst)):    print("   ", f, f"{os.path.getsize(os.path.join(dst,f))/1e6:.1f}MB")